# SOLID principles

- SRP - Single Responsibility Principle
  - only one reason to change
  - one responsibility/job
- Open/Closed Principle
  - open for extension, closed for modification
  - add features -> shouldn't change existing, tested code 
- LSP - Liskov Substitution Principle
  - superclass instance should be replaceable with subclass instances without breaking correctness
- ISP - Interface Segregation Principle
  - clients not depend on method they don't use
  - split large interfaces into smaller ones
- DIP - Dependency Inversion Principle
  - high lvl modules should not depend on low lvl modules
  - both should depend on abstractions

In [ ]:
from numbers import Real
import json


class Employee:
    def __init__(self, name, phone, salary):
        self.name = name
        self.phone = phone
        self.salary = salary

    @property
    def salary(self):
        return self._salary

    @salary.setter
    def salary(self, salary):
        if not isinstance(salary, Real):
            raise TypeError(f"salary must be a number not {type(salary).__name__}")

        if salary < 0:
            raise ValueError(f"Salary must be higher than 0 not {salary}")

        self._salary = salary

    # resp 1: business logic
    def net_salary(self, tax_rate=0.3):
        return self.salary * (1 - tax_rate)
    
    # resp 2: reporting
    def generate_report(self, report_type="txt"):
        if report_type == "txt":
            with open(f"{self.name}.txt", "w") as file:
                file.write("Employee card\n")
                file.write(
                    f"Name: {self.name}\nSalary: {self.salary}\nNet salary:{self.net_salary()}\nPhone: {self.phone}"
                )
        elif report_type == "json":
            with open(f"{self.name}.json", "w") as file:
                json.dump(
                    {
                        "name": self.name,
                        "phone": self.phone,
                        "salary": self.salary,
                        "net_salary": self.net_salary(),
                    }, file, indent=3
                )

    def __repr__(self):
        return f"Employee({self.name!r}, {self.phone!r}, {self.salary})"


e1 = Employee("Erika", "072-123156", 43000)
e1.generate_report("txt")

## Refactor to follow SOLID

In [12]:
from numbers import Real
import json
from abc import ABC, abstractmethod


class Employee:
    def __init__(self, name, phone, salary, tax_rate=0.3):
        self.name = name
        self.phone = phone
        self.salary = salary
        self.tax_rate = tax_rate

    @property
    def salary(self):
        return self._salary

    @salary.setter
    def salary(self, salary):
        if not isinstance(salary, Real):
            raise TypeError(f"salary must be a number not {type(salary).__name__}")

        if salary < 0:
            raise ValueError(f"Salary must be higher than 0 not {salary}")

        self._salary = salary

    @property
    def net_salary(self):
        return self.salary * (1 - self.tax_rate)

    def __repr__(self):
        return f"Employee({self.name!r}, {self.phone!r}, {self.salary})"


class Report(ABC):
    @abstractmethod
    def content(self, employee: Employee) -> str: ...


class TxtReport(Report):
    def content(self, employee):
        return (
            "Employee card\n"
            f"Name: {employee.name}\nSalary: {employee.salary}\nNet salary:{employee.net_salary}\nPhone: {employee.phone}"
        )


class JsonReport(Report):
    def content(self, employee):
        return json.dumps(
            {
                "name": employee.name,
                "phone": employee.phone,
                "salary": employee.salary,
                "net_salary": employee.net_salary,
            },
            indent=3,
        )


def export(text, path):
    with open(path, "w") as file:
        file.write(text)


e1 = Employee("Erika", "072-123156", 43000)

e1_txt_content = TxtReport().content(e1)
e1_json_content = JsonReport().content(e1)

export(e1_json_content,f"exports/{e1.name}.json")
export(e1_txt_content,f"exports/{e1.name}.txt")